# generar tripletas y embeddings

In [1]:
# -*- coding: utf-8 -*-
import os
import sys
import random
import json
import pandas as pd
import numpy as np
import torch
from types import SimpleNamespace

# Apuntar a la carpeta raíz donde está model.py
sys.path.append(r"C:\Users\56946\TuckER") 
from model import TuckER

# ============================================================
# 🔹 CONFIGURACIÓN GLOBAL
# ============================================================

# ⚠️ CARPETA SALIDA (MULTIRRELACIONAL)
OUTPUT_DATASET_DIR = r"../../data/dataset_2019_2020_2021_multirelacional_4d"
OUTPUT_EMBEDDINGS_DIR = r"../../notebooks/Experimento_warm_start/embeddings_4d_multirelacional"

DF_BASE = r"../../mis_scripts/dataframes_por_semestre"

# Pares de semestres (Cohortes)
PARES_SEMESTRES = [
    ("20191", "20192"), 
    ("20201", "20202"), 
    ("20211", "20212")
]

# Definición de Cursos (CRITERIO ESTRICTO)
CURSOS_PRIMER  = {"MA1101", "MA1001", "FI1000", "BT1211"}
CURSOS_SEGUNDO = {"MA1002", "MA1102", "FI1100", "CC1002"}
CURSOS_PERMITIDOS = CURSOS_PRIMER.union(CURSOS_SEGUNDO)

# Configuración Modelo
EDIM = 4  # ⚠️ SOLO 4 NOTAS
RDIM_DUMMY = 10 # Valor dummy inicial para multirrelacional
SPLIT_RATIO = [0.8, 0.1, 0.1]
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# 🔹 FASE 1: GENERACIÓN DE TRIPLETAS (Lógica Estricta)
# ============================================================

def normalizar_columnas(df):
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    df["CURSO"] = df["CURSO"].astype(str).str.strip().str.upper()
    df["ESTADO_CURSO"] = df["ESTADO_CURSO"].astype(str)
    return df

def determinar_relacion_multi(estado, nota_val):
    """
    Clasifica en 4 categorías según nota y estado.
    """
    estado = str(estado)
    
    # 1. Obtener valor numérico de la nota
    try:
        val = float(str(nota_val).replace(",", "."))
    except:
        val = None

    # 2. Prioridad Reprobación explícita
    if "Reprobado" in estado or "Eliminado" in estado:
        return "reprueba"
    
    # 3. Si no hay nota numérica, asumimos reprueba por defecto (seguridad)
    if val is None:
        return "reprueba"

    # 4. Clasificación por rangos (Multirrelacional)
    if val < 4.0:
        return "reprueba"
    elif val < 5.0:
        return "aprueba_4_5"
    elif val < 6.0:
        return "aprueba_5_6"
    else:
        return "aprueba_6_7"

def generar_tripletas():
    print("\n--- 🔨 FASE 1: GENERANDO TRIPLETAS MULTIRRELACIONALES (3 COHORTES) ---")
    os.makedirs(OUTPUT_DATASET_DIR, exist_ok=True)
    
    tripletas = []
    ids_validos_global = set() 

    for sem_ant, sem_act in PARES_SEMESTRES:
        ruta_prev = os.path.join(DF_BASE, f"df_{sem_ant}.csv")
        ruta_act  = os.path.join(DF_BASE, f"df_{sem_act}.csv")
        
        if not os.path.exists(ruta_prev) or not os.path.exists(ruta_act):
            print(f"⚠️ Saltando {sem_ant}->{sem_act}")
            continue

        print(f"   Procesando: {sem_ant} -> {sem_act}")
        df_prev = normalizar_columnas(pd.read_csv(ruta_prev, sep=";"))
        df_act  = normalizar_columnas(pd.read_csv(ruta_act,  sep=";"))

        # -----------------------------------------------------------
        # ⚠️ CRITERIO DE SELECCIÓN DE ALUMNOS (IDÉNTICO AL BINARIO)
        # -----------------------------------------------------------
        
        # 1. Filtro S1: Toman los 4 cursos fundamentales
        df_fund_prev = df_prev[df_prev["CURSO"].isin(CURSOS_PRIMER)]
        conteo = df_fund_prev.groupby("ID")["CURSO"].nunique()
        alumnos_cumplen_s1 = set(conteo[conteo == len(CURSOS_PRIMER)].index)
        
        # 2. Filtro S2: Toman AL MENOS 1 de los 8 cursos fundamentales
        df_fund_act = df_act[df_act["CURSO"].isin(CURSOS_PERMITIDOS)]
        alumnos_cumplen_s2 = set(df_fund_act["ID"].unique())
        
        # 3. INTERSECCIÓN
        alumnos_validos = alumnos_cumplen_s1.intersection(alumnos_cumplen_s2)
        
        print(f"      -> Cumplen S1 (4 ramos): {len(alumnos_cumplen_s1)}")
        print(f"      -> Cumplen S2 (>=1 ramo): {len(alumnos_cumplen_s2)}")
        print(f"      -> INTERSECCIÓN VÁLIDA: {len(alumnos_validos)}")
        
        ids_validos_global.update(alumnos_validos)

        # 4. Generar Tripletas
        df_filtrado = df_act[
            (df_act["ID"].isin(alumnos_validos)) &
            (df_act["CURSO"].isin(CURSOS_PERMITIDOS))
        ]

        count_local = 0
        for _, row in df_filtrado.iterrows():
            # Usamos la nueva función multirrelacional
            rel = determinar_relacion_multi(row["ESTADO_CURSO"], row["NOTA"])
            if rel:
                tripletas.append(f"{row['ID']}\t{rel}\t{row['CURSO']}")
                count_local += 1
        
        print(f"      -> Tripletas generadas: {count_local}")

    # Shuffle y Split
    random.shuffle(tripletas)
    n = len(tripletas)
    n_train = int(n * SPLIT_RATIO[0])
    n_valid = int(n * SPLIT_RATIO[1])
    
    train_data = tripletas[:n_train]
    valid_data = tripletas[n_train : n_train + n_valid]
    test_data  = tripletas[n_train + n_valid :]
    
    print(f"\n📊 Total Tripletas Multirrelacionales: {n}")
    
    def guardar(nombre, data):
        with open(os.path.join(OUTPUT_DATASET_DIR, nombre), 'w', encoding='utf-8') as f:
            f.write("\n".join(data) + "\n")
            
    guardar("train.txt", train_data)
    guardar("valid.txt", valid_data)
    guardar("test.txt", test_data)
    
    # -----------------------------------------------------------
    # ⚖️ BALANCEO PARA MULTIRRELACIONAL
    # (Duplicamos 'reprueba' para igualar a la suma del resto)
    # -----------------------------------------------------------
    reprobados = [t for t in train_data if "\treprueba\t" in t]
    otros = [t for t in train_data if "\treprueba\t" not in t] # Todo lo que sea aprueba_*
    
    if len(reprobados) > 0 and len(otros) > len(reprobados):
        factor = len(otros) // len(reprobados)
        resto = len(otros) % len(reprobados)
        train_bal = otros + (reprobados * factor) + reprobados[:resto]
        random.shuffle(train_bal)
        guardar("train_balanceado.txt", train_bal)
        print(f"   ⚖️ Balanceo: {len(reprobados)} reprobados orig -> {len(train_bal) - len(otros)} en train_bal")
    else:
        guardar("train_balanceado.txt", train_data)

    print(f"✅ Dataset guardado en: {OUTPUT_DATASET_DIR}")
    return ids_validos_global

# ============================================================
# 🔹 FASE 2: GENERACIÓN DE EMBEDDINGS 4D (SOLO NOTAS)
# ============================================================

def normalizar_nota(n):
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

def construir_matriz_notas(rutas_s1):
    df_list = []
    for r in rutas_s1:
        if os.path.exists(r):
            df_list.append(pd.read_csv(r, sep=";"))
    if not df_list: return pd.DataFrame()
            
    df_total = pd.concat(df_list, ignore_index=True)
    df_total = normalizar_columnas(df_total)
    
    # Filtrar solo fundamentales S1
    df_total = df_total[df_total["CURSO"].isin(CURSOS_PRIMER)]
    
    df_total['NOTA'] = pd.to_numeric(df_total['NOTA'], errors='coerce')
    pivot = df_total.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
    return pivot.reindex(columns=list(CURSOS_PRIMER))

def get_vocab_generated():
    entities = set()
    relations = set()
    for fname in ["train.txt", "valid.txt", "test.txt"]:
        path = os.path.join(OUTPUT_DATASET_DIR, fname)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                h, r, t = line.strip().split()
                entities.add(h); entities.add(t); relations.add(r)
    
    d = SimpleNamespace()
    d.entities = sorted(list(entities))
    d.relations = sorted(list(relations)) + [r + "_reverse" for r in sorted(list(relations))]
    d.entity_idxs = {e: i for i, e in enumerate(d.entities)}
    d.relation_idxs = {r: i for i, r in enumerate(d.relations)}
    return d

def generar_embeddings(ids_validos):
    print("\n--- 🧠 FASE 2: INICIALIZANDO EMBEDDINGS 4D (Solo Notas) ---")
    os.makedirs(OUTPUT_EMBEDDINGS_DIR, exist_ok=True)
    
    d = get_vocab_generated()
    print(f"   Vocabulario: {len(d.entities)} entidades.")
    
    # Cargar notas S1 de TODOS los años
    rutas_s1 = [
        os.path.join(DF_BASE, "df_20191.csv"),
        os.path.join(DF_BASE, "df_20201.csv"),
        os.path.join(DF_BASE, "df_20211.csv")
    ]
    tabla_notas = construir_matriz_notas(rutas_s1)
    
    # Inicializar TuckER con EDIM=4
    modelo = TuckER(d, EDIM, RDIM_DUMMY, input_dropout=0, hidden_dropout1=0, hidden_dropout2=0)
    
    count = 0
    with torch.no_grad():
        for entity, idx in d.entity_idxs.items():
            if entity in ids_validos and entity in tabla_notas.index:
                row = tabla_notas.loc[entity]
                
                # ⚠️ Solo 4 Notas Normalizadas [-1, 1]
                vec_notas = [normalizar_nota(row[c]) for c in CURSOS_PRIMER]
                
                final_vec = np.array(vec_notas, dtype=np.float32)
                
                if final_vec.shape[0] != 4:
                    print(f"⚠️ Error dimensión: {entity} tiene {final_vec.shape}")
                    continue

                modelo.E.weight[idx] = torch.tensor(final_vec)
                count += 1

    print(f"   ✅ Se inicializaron {count} alumnos con datos reales 4D.")
    
    path_pt = os.path.join(OUTPUT_EMBEDDINGS_DIR, "embeddings_inicializados_multi_4d.pt")
    path_json = os.path.join(OUTPUT_EMBEDDINGS_DIR, "vocabulario_multi_4d.json")
    
    torch.save(modelo.E.weight.data, path_pt)
    with open(path_json, 'w', encoding='utf-8') as f:
        json.dump({"entities": d.entities, "relations": d.relations}, f, indent=2)
        
    print(f"💾 Guardado: {path_pt}")
    print(f"💾 Guardado: {path_json}")

if __name__ == "__main__":
    ids_val = generar_tripletas()
    generar_embeddings(ids_val)


--- 🔨 FASE 1: GENERANDO TRIPLETAS MULTIRRELACIONALES (3 COHORTES) ---
   Procesando: 20191 -> 20192
      -> Cumplen S1 (4 ramos): 832
      -> Cumplen S2 (>=1 ramo): 802
      -> INTERSECCIÓN VÁLIDA: 790
      -> Tripletas generadas: 2961
   Procesando: 20201 -> 20202
      -> Cumplen S1 (4 ramos): 811
      -> Cumplen S2 (>=1 ramo): 877
      -> INTERSECCIÓN VÁLIDA: 787
      -> Tripletas generadas: 3021
   Procesando: 20211 -> 20212
      -> Cumplen S1 (4 ramos): 830
      -> Cumplen S2 (>=1 ramo): 869
      -> INTERSECCIÓN VÁLIDA: 783
      -> Tripletas generadas: 3023

📊 Total Tripletas Multirrelacionales: 9005
   ⚖️ Balanceo: 398 reprobados orig -> 6806 en train_bal
✅ Dataset guardado en: ../../data/dataset_2019_2020_2021_multirelacional_4d

--- 🧠 FASE 2: INICIALIZANDO EMBEDDINGS 4D (Solo Notas) ---
   Vocabulario: 2368 entidades.
   ✅ Se inicializaron 2353 alumnos con datos reales 4D.
💾 Guardado: ../../notebooks/Experimento_warm_start/embeddings_4d_multirelacional\embeddings_in

# Generar redes 

In [1]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# ⚙️ CONFIGURACIÓN 4D
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# 1. Vocabulario (Dataset Multirrelacional)
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

# 2. Resultados TuckER 4D
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# ⚠️ AJUSTA ESTE NOMBRE AL DE TU CARPETA 4D
RUN_PREFIX = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_multirelacional_patience400_5d"

# 3. Datos y Salida
BASE_DF_PATH  = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
SAVE_BASE     = r"C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_4d"

CURSOS_PRIMER = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO= ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PERMITIDOS = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = list(range(1, 17)) 

# =========================
# 🛠️ FUNCIONES
# =========================
def cargar_df_notas(path_csv):
    if not os.path.exists(path_csv): return pd.DataFrame()
    df = pd.read_csv(path_csv, sep=';')
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if not os.path.exists(path): continue
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1: entities.add(parts[0]); entities.add(parts[2])
    return sorted(list(entities))

def pick_state_dict(ckpt_loaded):
    if isinstance(ckpt_loaded, dict):
        if "model_state_dict" in ckpt_loaded: return ckpt_loaded["model_state_dict"]
        if "state_dict" in ckpt_loaded: return ckpt_loaded["state_dict"]
    return ckpt_loaded

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    return sd["E.weight"].detach().cpu()

def norm_nota(n):
    try:
        if pd.isna(n): return -1.0
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except: return -1.0

# ⚠️ FUNCIÓN 4D: SOLO NOTAS, SIN PUNTAJE
def construir_vectores_4d(df_sem1, df_sem2, cursos_primer, cursos_permitidos):
    # 1. Filtro
    df_fund = df_sem1[df_sem1['CURSO'].isin(cursos_primer)]
    conteo = df_fund.groupby('ID')['CURSO'].nunique()
    alumnos_4 = conteo[conteo == len(cursos_primer)].index

    df_sem2_filt = df_sem2[
        (df_sem2['ID'].isin(alumnos_4)) &
        (df_sem2['CURSO'].isin(cursos_permitidos))
    ]
    alumnos_validos = sorted(df_sem2_filt['ID'].unique())

    if not alumnos_validos: return pd.DataFrame()

    # 2. Matriz de Notas
    df_notas = df_sem1[df_sem1['ID'].isin(alumnos_validos) & df_sem1['CURSO'].isin(cursos_primer)].copy()
    
    pivot = df_notas.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
    pivot = pivot.reindex(columns=cursos_primer)
    pivot = pivot.applymap(norm_nota)
    
    return pivot

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def entrenar_predictor(X_data, Y_data, save_path, max_epochs=500, lr=1e-3, patience=100):
    X_train, X_test, Y_train, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)
    X_tr, X_val, Y_tr, Y_val          = train_test_split(X_train, Y_train, test_size=0.15, random_state=42)

    X_tr_t = torch.FloatTensor(X_tr); Y_tr_t = torch.FloatTensor(Y_tr)
    X_val_t= torch.FloatTensor(X_val);Y_val_t= torch.FloatTensor(Y_val)
    X_te_t = torch.FloatTensor(X_test);Y_te_t= torch.FloatTensor(Y_test)

    model = EmbeddingPredictor(X_tr.shape[1], Y_tr.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val, patience_counter = float('inf'), 0

    for epoch in range(1, max_epochs + 1):
        model.train(); optimizer.zero_grad()
        loss = criterion(model(X_tr_t), Y_tr_t)
        loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad(): vloss = criterion(model(X_val_t), Y_val_t)

        if vloss.item() < best_val - 1e-9:
            best_val = vloss.item()
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval()
    with torch.no_grad(): test_mse = criterion(model(X_te_t), Y_te_t).item()
    return test_mse

# =========================
# MAIN
# =========================
def main():
    if not os.path.exists(SAVE_BASE): os.makedirs(SAVE_BASE)
    print(f"📂 Guardando redes en: {SAVE_BASE}")
    print("=== Entrenando Redes MULTIRRELACIONALES 4D (3 Cohortes) ===")

    vocab = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab)}
    
    df_19_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20191.csv"))
    df_19_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20192.csv"))
    df_20_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20201.csv"))
    df_20_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20202.csv"))
    df_21_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20211.csv"))
    df_21_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20212.csv"))
    
    # Construir Vectores 4D (Solo Notas)
    print("   Construyendo vectores 4D...")
    vecs_19 = construir_vectores_4d(df_19_1, df_19_2, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    vecs_20 = construir_vectores_4d(df_20_1, df_20_2, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    vecs_21 = construir_vectores_4d(df_21_1, df_21_2, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    
    X_total_df = pd.concat([vecs_19, vecs_20, vecs_21], axis=0)
    X_total_df = X_total_df[~X_total_df.index.duplicated(keep='first')]
    
    print(f"-> Vectores X listos: {len(X_total_df)} alumnos. Dim={X_total_df.shape[1]}")

    resumen = []
    for rdim in RDIMS:
        run_dir   = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim))
        tucker_pt = os.path.join(run_dir, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_rdim{rdim}_multi_4d.pt")

        print(f"\n>> rdim={rdim}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ Pendiente/No existe: {tucker_pt}")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            alumnos_comunes = sorted(set(X_total_df.index).intersection(entity_idxs.keys()))
            if not alumnos_comunes:
                print("   ⚠️ Sin intersección.")
                continue
                
            X_data = X_total_df.loc[alumnos_comunes].values.astype(np.float32)
            idxs   = [entity_idxs[a] for a in alumnos_comunes]
            Y_data = E[idxs].numpy()
            
            mse = entrenar_predictor(X_data, Y_data, save_path)
            print(f"   ✅ Guardado. MSE: {mse:.6f}")
            resumen.append((rdim, mse))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")

    if resumen:
        pd.DataFrame(resumen, columns=["rdim", "mse"]).to_csv(os.path.join(SAVE_BASE, "resumen_multi_4d.csv"), index=False)
        print("\n✅ Proceso finalizado.")

if __name__ == "__main__":
    main()

📂 Guardando redes en: C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_4d
=== Entrenando Redes MULTIRRELACIONALES 4D (3 Cohortes) ===
   Construyendo vectores 4D...
-> Vectores X listos: 2351 alumnos. Dim=4

>> rdim=1
   ✅ Guardado. MSE: 0.010834

>> rdim=2
   ✅ Guardado. MSE: 0.010227

>> rdim=3
   ✅ Guardado. MSE: 0.008694

>> rdim=4
   ✅ Guardado. MSE: 0.008898

>> rdim=5
   ✅ Guardado. MSE: 0.012502

>> rdim=6
   ✅ Guardado. MSE: 0.010592

>> rdim=7
   ✅ Guardado. MSE: 0.016295

>> rdim=8
   ✅ Guardado. MSE: 0.013583

>> rdim=9
   ✅ Guardado. MSE: 0.008254

>> rdim=10
   ✅ Guardado. MSE: 0.011672

>> rdim=11
   ✅ Guardado. MSE: 0.011227

>> rdim=12
   ✅ Guardado. MSE: 0.010058

>> rdim=13
   ✅ Guardado. MSE: 0.009292

>> rdim=14
   ✅ Guardado. MSE: 0.009971

>> rdim=15
   ✅ Guardado. MSE: 0.011265

>> rdim=16
   ✅ Guardado. MSE: 0.011156

✅ Proceso finalizado.


# probar modelo colapsando relación reprueba(solo evalua reprueba)

In [2]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN 4D (3 COHORTES)
# ============================================================

# Vocabulario del Dataset Multirrelacional (4D)
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

# Ruta base de resultados TuckER
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# ⚠️ IMPORTANTE: Usamos el prefijo que indicaste en la configuración.
# Verifica que esta carpeta exista en 'results'. Si entrenaste TuckER 4D con otro nombre, cámbialo aquí.
RUN_PREFIX = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_multirelacional_patience400_5d"

# Ruta donde guardaste los Predictores Neuronales 4D (.pt)
PRED_DIR_BASE = r"C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_4d"

# Datos Evaluación (2021)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_S1 = os.path.join(BASE_PATH, "df_20211.csv")
CSV_S2 = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]
RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 MODELOS
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    """
    Versión corregida para evitar el error 'NoneType is not subscriptable'.
    """
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
        # Si es un dict pero no tiene esas llaves, asumimos que ya son los pesos
        return ckpt
    return ckpt

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(state))
    model.to(DEVICE).eval()
    return model

def load_tucker_weights(path):
    raw = torch.load(path, map_location=DEVICE)
    state = pick_state_dict(raw)
    return state["E.weight"].to(DEVICE), state["R.weight"].to(DEVICE), state["W"].to(DEVICE)

def get_vocab(data_dir):
    entities, relations = set(), set()
    # Leemos todos los archivos para asegurar vocabulario completo
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h); entities.add(t); relations.add(r)
    return SimpleNamespace(
        entities=sorted(list(entities)),
        relations=sorted(list(relations) + [r+"_reverse" for r in relations]),
        entity_idxs={e:i for i,e in enumerate(sorted(list(entities)))},
        relation_idxs={r:i for i,r in enumerate(sorted(list(relations) + [r+"_reverse" for r in relations]))}
    )

def find_rel_idx(vocab, hint):
    cands = [r for r in vocab.relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return vocab.relation_idxs[cands[0]]

@torch.no_grad()
def tucker_score(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    # Manejo de layouts de TuckER (por si W cambió dimensiones)
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    else:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ============================================================
# 🔹 PROCESAMIENTO 4D (SOLO NOTAS)
# ============================================================

def norm_nota(n):
    # Normalización idéntica al entrenamiento: (n - 4) / 3 -> [-1, 1]
    if pd.isna(n): return -1.0
    try:
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except:
        return -1.0

def get_input_vector_4d(df_s1, aid, cursos_primer):
    # Solo construimos vector de tamaño 4 (Notas)
    subset = df_s1[(df_s1['ID'] == aid) & (df_s1['CURSO'].isin(cursos_primer))]
    vec = [-1.0] * len(cursos_primer)
    
    if not subset.empty:
        notas_dict = dict(zip(subset['CURSO'], subset['NOTA']))
        for i, c in enumerate(cursos_primer):
            if c in notas_dict:
                vec[i] = norm_nota(notas_dict[c])
    
    # Retornamos Tensor (1, 4)
    return torch.tensor(vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- EVALUACIÓN FINAL 4D (SOLO NOTAS) ---")
    
    # Cargar Dataframes
    df_s1 = pd.read_csv(CSV_S1, sep=';')
    df_s2 = pd.read_csv(CSV_S2, sep=';')
    for df in (df_s1, df_s2):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # Cargar Vocabulario
    vocab = get_vocab(DATA_DIR)
    
    # Filtrar alumnos válidos (que tengan notas en S1)
    alumnos_s1 = df_s1.groupby("ID")["CURSO"].apply(set)
    ids_ok = alumnos_s1[alumnos_s1.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index
    
    # DataFrame de evaluación (S2)
    df_eval = df_s2[
        (df_s2['ID'].isin(ids_ok)) & 
        (df_s2['CURSO'].isin(CURSOS_PRED_ALL))
    ].copy()
    
    def get_real_rel(row):
        n = pd.to_numeric(row['NOTA'], errors='coerce')
        st = str(row['ESTADO_CURSO'])
        if "Reprobado" in st or pd.isna(n) or n < 4.0: return "reprueba"
        if n < 5.0: return "aprueba_4_5"
        if n < 6.0: return "aprueba_5_6"
        return "aprueba_6_7"
    
    df_eval["REL_REAL"] = df_eval.apply(get_real_rel, axis=1)
    
    print(f"Total Evaluaciones: {len(df_eval)}")

    for rdim in RDIMS:
        print(f"\n>> Evaluando rdim={rdim}")
        
        # Rutas de Archivos
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        # Buscamos el predictor 'multi_4d'
        pred_pt = os.path.join(PRED_DIR_BASE, f"best_predictor_rdim{rdim}_multi_4d.pt")
        
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ Falta TuckER: {tucker_pt}")
            continue
        if not os.path.exists(pred_pt):
            print(f"   ⚠️ Falta Predictor: {pred_pt}")
            continue
            
        # Cargar Modelos
        E, R, W = load_tucker_weights(tucker_pt)
        d1 = E.shape[1]
        
        # Mapear relaciones
        rel_map = {}
        for rname in RELACIONES:
            idx = find_rel_idx(vocab, rname)
            if idx is not None: rel_map[rname] = idx
            
        # Cargar Predictor con Input=4
        predictor = load_predictor(pred_pt, input_size=4, out_dim=d1)
        
        # Precalcular Embeddings (Cache)
        ehat_cache = {}
        unique_ids = df_eval["ID"].unique()
        for uid in unique_ids:
            # Usamos vector 4D (sin puntaje)
            x = get_input_vector_4d(df_s1, uid, CURSOS_PRIMER)
            with torch.no_grad(): ehat_cache[uid] = predictor(x).squeeze(0)
            
        # Evaluar Predicciones
        y_true, y_pred = [], []
        
        for _, row in df_eval.iterrows():
            uid, cur, rel_real = row["ID"], row["CURSO"], row["REL_REAL"]
            if cur not in vocab.entity_idxs: continue
            
            t_idx = vocab.entity_idxs[cur]
            e_hat = ehat_cache[uid]
            
            # Ranking de relaciones
            scores = {r: tucker_score(e_hat, R, W, E, idx, t_idx) for r, idx in rel_map.items()}
            rel_pred = max(scores, key=scores.get)
            
            # Binarizar (1 = Riesgo/Reprueba)
            y_true.append(1 if "reprueba" in rel_real else 0)
            y_pred.append(1 if "reprueba" in rel_pred else 0)
            
        # Métricas Finales
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp+fn) > 0 else 0
        prec = tp / (tp + fp) if (tp+fp) > 0 else 0
        acc = (tp + tn) / len(y_true)
        
        print("-" * 50)
        print(f"Riesgo Real (TP+FN): {tp+fn}")
        print(f"Recall:    {recall:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Matriz: TP={tp}, FN={fn}, FP={fp}, TN={tn}")

if __name__ == "__main__":
    main()


--- EVALUACIÓN FINAL 4D (SOLO NOTAS) ---
Total Evaluaciones: 3023

>> Evaluando rdim=1
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.2318
Precision: 0.0405
Accuracy:  0.5180
Matriz: TP=54, FN=179, FP=1278, TN=1512

>> Evaluando rdim=2
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.5536
Precision: 0.0843
Accuracy:  0.5022
Matriz: TP=129, FN=104, FP=1401, TN=1389

>> Evaluando rdim=3
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.3863
Precision: 0.0600
Accuracy:  0.4863
Matriz: TP=90, FN=143, FP=1410, TN=1380

>> Evaluando rdim=4
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    1.0000
Precision: 0.0771
Accuracy:  0.0771
Matriz: TP=233, FN=0, FP=2790, TN=0

>> Evaluando rdim=5
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    1.0000
Precision: 0.0771
Accuracy:  0.0771
Matriz: TP=233, FN=0,

# Colapsando aprueba con 4,5 y reprueba

In [3]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN 4D (3 COHORTES)
# ============================================================

DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

RESULTS_BASE = r"C:\Users\56946\TuckER\results"
# ⚠️ Ajustar al nombre exacto de tu carpeta de resultados 4D
RUN_PREFIX = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_multirelacional_patience400_5d"

PRED_DIR_BASE = r"C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_4d"

BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_S1 = os.path.join(BASE_PATH, "df_20211.csv")
CSV_S2 = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]
RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 MODELOS Y UTILIDADES
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
        return ckpt
    return ckpt

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(state))
    model.to(DEVICE).eval()
    return model

def load_tucker_weights(path):
    raw = torch.load(path, map_location=DEVICE)
    state = pick_state_dict(raw)
    return state["E.weight"].to(DEVICE), state["R.weight"].to(DEVICE), state["W"].to(DEVICE)

def get_vocab(data_dir):
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h); entities.add(t); relations.add(r)
    return SimpleNamespace(
        entities=sorted(list(entities)),
        relations=sorted(list(relations) + [r+"_reverse" for r in relations]),
        entity_idxs={e:i for i,e in enumerate(sorted(list(entities)))},
        relation_idxs={r:i for i,r in enumerate(sorted(list(relations) + [r+"_reverse" for r in relations]))}
    )

def find_rel_idx(vocab, hint):
    cands = [r for r in vocab.relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return vocab.relation_idxs[cands[0]]

@torch.no_grad()
def tucker_score(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    else:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ============================================================
# 🔹 CONSTRUCCIÓN DE VECTORES (4D - SOLO NOTAS)
# ============================================================
def norm_nota(n):
    if pd.isna(n): return -1.0
    try:
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except: return -1.0

def get_input_vector_4d(df_s1, aid, cursos_primer):
    # Solo notas, tamaño 4
    subset = df_s1[(df_s1['ID'] == aid) & (df_s1['CURSO'].isin(cursos_primer))]
    vec = [-1.0] * len(cursos_primer)
    if not subset.empty:
        notas_dict = dict(zip(subset['CURSO'], subset['NOTA']))
        for i, c in enumerate(cursos_primer):
            if c in notas_dict:
                vec[i] = norm_nota(notas_dict[c])
    return torch.tensor(vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 LÓGICA RIESGO AMPLIADO
# ============================================================
def es_riesgo_ampliado(relacion_str):
    rel = relacion_str.lower()
    return "reprueba" in rel or "aprueba_4_5" in rel

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- EVALUACIÓN FINAL 4D (RIESGO AMPLIADO) ---")
    
    df_s1 = pd.read_csv(CSV_S1, sep=';')
    df_s2 = pd.read_csv(CSV_S2, sep=';')
    for df in (df_s1, df_s2):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    vocab = get_vocab(DATA_DIR)
    
    alumnos_s1 = df_s1.groupby("ID")["CURSO"].apply(set)
    ids_ok = alumnos_s1[alumnos_s1.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index
    
    df_eval = df_s2[
        (df_s2['ID'].isin(ids_ok)) & 
        (df_s2['CURSO'].isin(CURSOS_PRED_ALL))
    ].copy()
    
    def get_real_rel(row):
        n = pd.to_numeric(row['NOTA'], errors='coerce')
        st = str(row['ESTADO_CURSO'])
        if "Reprobado" in st or pd.isna(n) or n < 4.0: return "reprueba"
        if n < 5.0: return "aprueba_4_5"
        if n < 6.0: return "aprueba_5_6"
        return "aprueba_6_7"
    
    df_eval["REL_REAL"] = df_eval.apply(get_real_rel, axis=1)
    print(f"Evaluaciones totales: {len(df_eval)}")

    for rdim in RDIMS:
        print(f"\n>> Evaluando rdim={rdim}")
        
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        pred_pt = os.path.join(PRED_DIR_BASE, f"best_predictor_rdim{rdim}_multi_4d.pt")
        
        if not os.path.exists(tucker_pt) or not os.path.exists(pred_pt):
            print("   ⏩ Saltando (Faltan archivos)")
            continue
            
        E, R, W = load_tucker_weights(tucker_pt)
        d1 = E.shape[1]
        
        rel_map = {}
        for rname in RELACIONES:
            idx = find_rel_idx(vocab, rname)
            if idx is not None: rel_map[rname] = idx
            
        predictor = load_predictor(pred_pt, input_size=4, out_dim=d1)
        
        ehat_cache = {}
        for uid in df_eval["ID"].unique():
            x = get_input_vector_4d(df_s1, uid, CURSOS_PRIMER)
            with torch.no_grad(): ehat_cache[uid] = predictor(x).squeeze(0)
            
        y_true, y_pred = [], []
        
        for _, row in df_eval.iterrows():
            uid, cur, rel_real = row["ID"], row["CURSO"], row["REL_REAL"]
            if cur not in vocab.entity_idxs: continue
            
            t_idx = vocab.entity_idxs[cur]
            e_hat = ehat_cache[uid]
            
            scores = {r: tucker_score(e_hat, R, W, E, idx, t_idx) for r, idx in rel_map.items()}
            rel_pred = max(scores, key=scores.get)
            
            # ⚠️ COLAPSO RIESGO AMPLIADO
            y_true.append(1 if es_riesgo_ampliado(rel_real) else 0)
            y_pred.append(1 if es_riesgo_ampliado(rel_pred) else 0)
            
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp+fn) > 0 else 0
        prec = tp / (tp + fp) if (tp+fp) > 0 else 0
        acc = (tp + tn) / len(y_true)
        
        print("-" * 60)
        print(f"Accuracy Global          : {acc:.4f}")
        print(f"Total REALES en Riesgo   : {tp+fn} (Repro + Nota 4-5)")
        print("-" * 60)
        print(f"✅ RECALL (Sensibilidad) : {recall:.4f}")
        print(f"   (Detectamos {tp} de {tp+fn} casos)")
        print("-" * 60)
        print(f"🎯 PRECISION             : {prec:.4f}")
        print(f"Matriz: TP={tp}, FN={fn}, FP={fp}, TN={tn}")

if __name__ == "__main__":
    main()


--- EVALUACIÓN FINAL 4D (RIESGO AMPLIADO) ---
Evaluaciones totales: 3023

>> Evaluando rdim=1
------------------------------------------------------------
Accuracy Global          : 0.4707
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.3342
   (Detectamos 270 de 808 casos)
------------------------------------------------------------
🎯 PRECISION             : 0.2027
Matriz: TP=270, FN=538, FP=1062, TN=1153

>> Evaluando rdim=2
------------------------------------------------------------
Accuracy Global          : 0.4152
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.8589
   (Detectamos 694 de 808 casos)
------------------------------------------------------------
🎯 PRECISION             : 0.2956
Matriz: TP=694, FN=114, FP=1654, TN=561

>> Evaluando rdim=3
--------------------------------------------------